In [ ]:
import requests
import json
import re
from datetime import date, datetime, timedelta
from requests.auth import HTTPBasicAuth
from IPython.display import display
import ipywidgets as widgets
from collections import defaultdict

# ═══════════════════════════════════════
# ÖNCƏ BUNLARI DOLDUR
# ═══════════════════════════════════════
GROQ_API_KEY   = "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"
JIRA_URL       = "https://alinurkhanim.atlassian.net"
JIRA_EMAIL     = "ali.nurkhanim@gmail.com"
JIRA_API_TOKEN = "ATATT3xFfGF0o3L2AfGsZcA4OE1YJrH9NJCCFecK-OSHeXf_4NSn-yhVombw5U3GgyA9UONdP-jdrz511L77t0364jLG4Hcth4W1Y1GvNLPcOCsh68L0zgM7JH0dUorn76fG3WULqQ7JTPNypXNdSgoATVBuNdKt1CzI9OEpT55lPJuttMuiZEU=790413D1"
JIRA_PROJECT   = "KAN"
# ═══════════════════════════════════════


def ai_parse_çoxlu(mətnlər):
    sətir_mətni = "\n".join(
        [f"{i+1}. {m}" for i, m in enumerate(mətnlər)]
    )

    prompt = f"""Aşağıdakı siyahıda hər nömrəli sətir ayrıca bir tapşırıqdır.
Hər tapşırıq üçün məlumatları çıxar.

QAYDALAR:
- task_adi: mətnde "tapşırıq", "task", "iş" sözlərinin yanında yazılan 
  hissəni götür. Yalnız mətndə olan sözlərdən istifadə et, əlavə etmə.
- icraçı: tapşırığı edəcək şəxsin adı
- deadline: bitmə tarixi YYYY-MM-DD formatında
  (bugün {date.today().strftime("%Y-%m-%d")}-dir)
  ("sabah" → +1 gün, "bu həftə" → +7 gün,
   ay adı deyilibsə həmin ayın sonu)
- story_points: tapşırığın həcmi Fibonacci ilə (1,2,3,5,8,13)
  kiçik/sadə iş → 1-2
  orta iş → 3-5
  böyük/mürəkkəb iş → 8-13
- asılılıq: hansı tapşırıqdan asılıdır (yoxdursa: null)
- kpi: mətnde "hədəf" sözündən sonra gələn hissəni olduğu kimi götür 
  və nömrələ. Əgər "hədəf" sözü yoxdursa — kpi sahəsini boş burax ("").
  Heç nə uydurma, yalnız mətndə yazılanı götür.
- prioritet: aşağıdakı 3 faktorа əsasən özün hesabla:
  1. Təsir: KPI-dakı son nəticəyə bax — nəyə çatılacaq?
     Biznes/istifadəçiyə kritik təsir → yüksək bal
     Daxili/kiçik təsir → aşağı bal
  2. Deadline: bugündən neçə gün var?
     0-3 gün → yüksək bal
     4-7 gün → orta bal
     7+ gün  → aşağı bal
  3. Çətinlik: story_points-ə əsasən
     8-13 → yüksək bal
     3-5  → orta bal
     1-2  → aşağı bal
  3 faktorun cəminə görə High / Medium / Low qaytar
- prioritet_səbəb: niyə bu prioriteti verdiyini 1 cümlə ilə izah et

Yalnız JSON array qaytar, başqa heç nə yazma.
Tapşırıqların sırası eyni olmalıdır:
[
  {{
    "task_adi": "...",
    "icraçı": "...",
    "deadline": "YYYY-MM-DD",
    "story_points": 3,
    "asılılıq": "..." və ya null,
    "kpi": "...",
    "prioritet": "High" və ya "Medium" və ya "Low",
    "prioritet_səbəb": "..."
  }},
  ...
]

Tapşırıqlar:
{sətir_mətni}"""

    cavab = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {GROQ_API_KEY}",
            "Content-Type": "application/json"
        },
        json={
            "model": "llama-3.3-70b-versatile",
            "max_tokens": 1500,
            "messages": [{"role": "user", "content": prompt}]
        }
    )

    if cavab.status_code != 200:
        print(f"❌ AI xətası: {cavab.status_code}")
        return None

    məzmun = cavab.json()["choices"][0]["message"]["content"].strip()
    məzmun = re.sub(r"```json|```", "", məzmun).strip()

    try:
        nəticə = json.loads(məzmun)
        if isinstance(nəticə, dict):
            nəticə = [nəticə]
        return nəticə
    except:
        print(f"❌ JSON parse xətası: {məzmun}")
        return None


def jira_issue_yarat(task_adi, icraçı, deadline, asılılıq, prioritet, story_points=1, kpi=""):
    auth    = HTTPBasicAuth(JIRA_EMAIL, JIRA_API_TOKEN)
    headers = {
        "Accept": "application/json",
        "Content-Type": "application/json"
    }
    bugun = date.today().strftime("%Y-%m-%d")
    kpi_sətirləri = kpi.split("\n") if kpi else ["Qeyd edilməyib"]

    data = {
        "fields": {
            "project":           {"key": JIRA_PROJECT},
            "summary":           task_adi,
            "description": {
                "type": "doc", "version": 1,
                "content": [
                    {"type": "paragraph", "content": [{"type": "text", "text": f"İcraçı: {icraçı}"}]},
                    {"type": "paragraph", "content": [{"type": "text", "text": f"Asılılıq: {asılılıq if asılılıq else 'Yoxdur'}"}]},
                    {"type": "paragraph", "content": [{"type": "text", "text": "KPI:"}]},
                ] + [
                    {"type": "paragraph", "content": [{"type": "text", "text": sətir}]}
                    for sətir in kpi_sətirləri if sətir.strip()
                ]
            },
            "issuetype":         {"name": "Task"},
            "duedate":           deadline,
            "customfield_10015": bugun,
            "priority":          {"name": prioritet},
            "customfield_10016": story_points,
            "customfield_10092": kpi
        }
    }

    cavab = requests.post(
        f"{JIRA_URL}/rest/api/3/issue",
        headers=headers,
        auth=HTTPBasicAuth(JIRA_EMAIL, JIRA_API_TOKEN),
        data=json.dumps(data)
    )

    if cavab.status_code == 201:
        return cavab.json()["key"]
    else:
        print(f"❌ Jira xətası: {cavab.status_code}")
        print(cavab.text)
        return None


# ═══════════════════════════════════════
# RİSK DETECTION
# ═══════════════════════════════════════

def jira_tapşırıqları_çək(done_daxil=False):
    auth    = HTTPBasicAuth(JIRA_EMAIL, JIRA_API_TOKEN)
    headers = {"Accept": "application/json", "Content-Type": "application/json"}
    statuslar = '"To Do", "In Progress", "Done"' if done_daxil else '"To Do", "In Progress"'
    jql     = f'project = {JIRA_PROJECT} AND status in ({statuslar}) ORDER BY duedate ASC'

    cavab = requests.post(
        f"{JIRA_URL}/rest/api/3/search/jql",
        headers=headers,
        auth=auth,
        json={
            "jql": jql,
            "maxResults": 100,
            "fields": ["summary", "status", "duedate", "assignee", "customfield_10016", "priority"]
        }
    )

    if cavab.status_code != 200:
        return None, f"Jira xətası: {cavab.status_code}"

    issues      = cavab.json().get("issues", [])
    tapşırıqlar = []

    for issue in issues:
        f        = issue["fields"]
        assignee = f.get("assignee")
        icraçı   = assignee["displayName"] if assignee else "Təyin edilməyib"

        deadline_str = f.get("duedate")
        deadline     = datetime.strptime(deadline_str, "%Y-%m-%d").date() if deadline_str else None
        story_points = f.get("customfield_10016") or 0

        tapşırıqlar.append({
            "key":          issue["key"],
            "ad":           f.get("summary", ""),
            "status":       f["status"]["name"],
            "deadline":     deadline,
            "icraçı":       icraçı,
            "story_points": story_points,
            "prioritet":    f.get("priority", {}).get("name", "Medium") if f.get("priority") else "Medium"
        })

    return tapşırıqlar, None


def gecikmə_riski_hesabla(tapşırıqlar):
    """
    Yalnız Jira məlumatlarına əsasən gecikmə riskini hesablayır.
    Qaydalar:
      - deadline keçib + hər hansı status      → Kritik
      - deadline bu gün və ya sabah             → Yüksək
      - deadline 3 gün + To Do                 → Yüksək
      - deadline 3 gün + In Progress           → Orta
      - deadline 7 gün + To Do                 → Orta
      - qalanlar                               → Normal
      - deadline yoxdur                        → Naməlum
    """
    bugun    = date.today()
    nəticələr = []

    for t in tapşırıqlar:
        if not t["deadline"]:
            risk     = "naməlum"
            səbəb    = "Deadline təyin edilməyib"
            qalan    = None
        else:
            qalan = (t["deadline"] - bugun).days

            if qalan < 0:
                risk  = "kritik"
                səbəb = f"Deadline {abs(qalan)} gün əvvəl keçib"
            elif qalan <= 1:
                risk  = "yüksək"
                səbəb = "Bu gün" if qalan == 0 else "Sabah son gündür"
            elif qalan <= 3 and t["status"] == "To Do":
                risk  = "yüksək"
                səbəb = f"{qalan} gün qalıb, hələ başlanmayıb"
            elif qalan <= 3 and t["status"] == "In Progress":
                risk  = "orta"
                səbəb = f"{qalan} gün qalıb, icra gedir"
            elif qalan <= 7 and t["status"] == "To Do":
                risk  = "orta"
                səbəb = f"{qalan} gün qalıb, hələ başlanmayıb"
            else:
                risk  = "normal"
                səbəb = f"{qalan} gün qalıb"

        nəticələr.append({**t, "risk": risk, "səbəb": səbəb, "qalan": qalan})

    # Sırala: kritik → yüksək → orta → normal → naməlum
    sıra = {"kritik": 0, "yüksək": 1, "orta": 2, "normal": 3, "naməlum": 4}
    nəticələr.sort(key=lambda x: sıra.get(x["risk"], 5))
    return nəticələr


def overload_riski_hesabla(tapşırıqlar):
    """
    Yalnız Jira məlumatlarına əsasən resurs overload hesablayır.
    Qayda: 1 icraçı üçün 5+ aktiv tapşırıq → Overload
    """
    icraçı_map = defaultdict(list)
    for t in tapşırıqlar:
        icraçı_map[t["icraçı"]].append(t)

    nəticələr = []
    for icraçı, siyahı in icraçı_map.items():
        say = len(siyahı)
        if say >= 5:
            səviyyə = "overload"
            mətn    = f"{say} aktiv tapşırıq — həddindən artıq yüklüdür"
        elif say >= 3:
            səviyyə = "yüklü"
            mətn    = f"{say} aktiv tapşırıq — diqqət tələb edir"
        else:
            səviyyə = "normal"
            mətn    = f"{say} aktiv tapşırıq"

        nəticələr.append({
            "icraçı":      icraçı,
            "say":         say,
            "səviyyə":     səviyyə,
            "mətn":        mətn,
            "tapşırıqlar": siyahı
        })

    nəticələr = [
        o for o in nəticələr
        if o["say"] >= 5 and o["icraçı"] != "Təyin edilməyib"
    ]
    nəticələr.sort(key=lambda x: x["say"], reverse=True)
    return nəticələr


def gecikmə_kartı_yarat(t):
    risk_konfiq = {
        "kritik":  {"rəng": "#c0392b", "ikona": "🔴", "başlıq": "KRİTİK"},
        "yüksək":  {"rəng": "#e74c3c", "ikona": "🔴", "başlıq": "YÜKSƏK"},
        "orta":    {"rəng": "#f39c12", "ikona": "🟡", "başlıq": "ORTA"},
        "normal":  {"rəng": "#27ae60", "ikona": "🟢", "başlıq": "NORMAL"},
        "naməlum": {"rəng": "#95a5a6", "ikona": "⚪", "başlıq": "NAMƏLUM"},
    }
    k        = risk_konfiq.get(t["risk"], risk_konfiq["naməlum"])
    deadline = t["deadline"].strftime("%d.%m.%Y") if t["deadline"] else "—"
    status   = t["status"]

    html = (
        f"<div style='border-left:4px solid {k['rəng']};padding:10px 14px;"
        f"margin:6px 0;background:#fafafa;border-radius:4px;font-size:13px'>"
        f"<div style='display:flex;justify-content:space-between;align-items:center'>"
        f"<b style='font-size:14px'>{k['ikona']} {t['key']} — {t['ad']}</b>"
        f"<span style='background:{k['rəng']};color:white;padding:2px 8px;"
        f"border-radius:10px;font-size:11px'>{k['başlıq']}</span></div>"
        f"<div style='margin-top:6px;color:#555'>"
        f"👤 <b>İcraçı:</b> {t['icraçı']} &nbsp;|&nbsp; "
        f"📅 <b>Deadline:</b> {deadline} &nbsp;|&nbsp; "
        f"📋 <b>Status:</b> {status}</div>"
        f"<div style='margin-top:4px;color:{k['rəng']}'><b>⚠️ Səbəb:</b> {t['səbəb']}</div>"
        f"</div>"
    )
    return widgets.HTML(html)


def overload_kartı_yarat(o):
    səviyyə_konfiq = {
        "overload": {"rəng": "#e74c3c", "ikona": "🔴", "başlıq": "OVERLOAD"},
        "yüklü":    {"rəng": "#f39c12", "ikona": "🟡", "başlıq": "YÜKLÜ"},
        "normal":   {"rəng": "#27ae60", "ikona": "🟢", "başlıq": "NORMAL"},
    }
    k = səviyyə_konfiq.get(o["səviyyə"], səviyyə_konfiq["normal"])

    tapşırıq_siyahısı = "".join([
        f"<div style='padding:2px 0;color:#555'>• {t['key']} — {t['ad'][:50]}</div>"
        for t in o["tapşırıqlar"]
    ])

    html = (
        f"<div style='border-left:4px solid {k['rəng']};padding:10px 14px;"
        f"margin:6px 0;background:#fafafa;border-radius:4px;font-size:13px'>"
        f"<div style='display:flex;justify-content:space-between;align-items:center'>"
        f"<b style='font-size:14px'>{k['ikona']} {o['icraçı']}</b>"
        f"<span style='background:{k['rəng']};color:white;padding:2px 8px;"
        f"border-radius:10px;font-size:11px'>{k['başlıq']}</span></div>"
        f"<div style='margin-top:6px;color:#555'>{o['mətn']}</div>"
        f"<div style='margin-top:6px'>{tapşırıq_siyahısı}</div>"
        f"</div>"
    )
    return widgets.HTML(html)


def agent():
    bugun  = date.today().strftime("%Y-%m-%d")
    style  = {"description_width": "150px"}
    layout = widgets.Layout(width="500px")

    w_giriş = widgets.Textarea(
        value="",
        description="✏️ Tapşırıqlar:",
        placeholder=(
            "Hər sətirə bir tapşırıq yazın:\n\n"
            "Aytən üçün hesabat yazma tapşırığı var. Bitmə tarixi 20 iyun.\n"
            "Elnur loqo fayllarını yeniləməlidir. Bitmə tarixi 25 iyun.\n"
            "Kamran klient görüşü hazırlamalıdır. Vacibdir. 18 iyun."
        ),
        style=style,
        layout=widgets.Layout(width="500px", height="140px")
    )

    analiz_btn   = widgets.Button(
        description="🧠 AI ilə Analiz Et",
        button_style="info",
        layout=widgets.Layout(width="200px", height="38px")
    )
    gecikmə_btn  = widgets.Button(
        description="⏰ Gecikmə Riski",
        button_style="warning",
        layout=widgets.Layout(width="170px", height="38px")
    )
    overload_btn = widgets.Button(
        description="👤 Overload Riski",
        button_style="danger",
        layout=widgets.Layout(width="170px", height="38px")
    )
    status_btn = widgets.Button(
        description="📊 Status Report",
        button_style="info",
        layout=widgets.Layout(width="170px", height="38px")
    )

    analiz_out   = widgets.Output()
    gecikmə_out  = widgets.Output()
    overload_out = widgets.Output()
    status_out   = widgets.Output()
    formlar_box  = widgets.VBox([])
    göndər_box   = widgets.VBox([])

    # ── Gecikmə riski düyməsi ────────────────────────
    _gecikmə_nəticələr = []

    def gecikmə_göstər(filtr="hamısı"):
        if filtr == "hamısı":
            görünən = [t for t in _gecikmə_nəticələr if t["risk"] != "normal"]
        else:
            görünən = [t for t in _gecikmə_nəticələr if t["risk"] == filtr]

        filtr_btn_stil = {
            "hamısı": "info",
            "kritik": "danger",
            "yüksək": "danger",
            "orta":   "warning",
        }

        filtr_box = widgets.HBox([
            widgets.HTML("<b style='line-height:32px;margin-right:8px'>Filtr:</b>"),
            *[
                widgets.Button(
                    description=ad,
                    button_style=("primary" if (f == filtr) else ""),
                    layout=widgets.Layout(width="100px", height="32px")
                )
                for f, ad in [
                    ("hamısı", "🔍 Hamısı"),
                    ("kritik", "🔴 Kritik"),
                    ("yüksək", "🔴 Yüksək"),
                    ("orta",   "🟡 Orta"),
                ]
            ]
        ], layout=widgets.Layout(margin="8px 0"))

        # Filtr düymələrinə click bağla
        for btn, (f, _) in zip(filtr_box.children[1:], [
            ("hamısı", ""), ("kritik", ""), ("yüksək", ""), ("orta", "")
        ]):
            def _click(_, f=f):
                gecikmə_göstər(f)
            btn.on_click(_click)

        say    = len(görünən)
        başlıq = widgets.HTML(
            f"<h4 style='margin:8px 0'>⏰ Gecikmə Riski — {say} tapşırıq</h4>"
        )

        if say == 0:
            məzmun = [başlıq, filtr_box,
                      widgets.HTML("<p style='color:#27ae60'>✅ Bu kateqoriyada tapşırıq yoxdur.</p>")]
        else:
            məzmun = [başlıq, filtr_box] + [gecikmə_kartı_yarat(t) for t in görünən]

        gecikmə_out.clear_output()
        with gecikmə_out:
            display(widgets.VBox(məzmun))

    def gecikmə_yoxla(_):
        nonlocal _gecikmə_nəticələr
        analiz_out.clear_output()
        overload_out.clear_output()
        status_out.clear_output()
        formlar_box.children = []
        göndər_box.children = []
        with gecikmə_out:
            gecikmə_out.clear_output()
            print("⏰ Jira-dan tapşırıqlar yüklənir...")

        tapşırıqlar, xəta = jira_tapşırıqları_çək()
        if xəta:
            with gecikmə_out:
                gecikmə_out.clear_output()
                print(f"❌ {xəta}")
            return
        if not tapşırıqlar:
            with gecikmə_out:
                gecikmə_out.clear_output()
                print("✅ Aktiv tapşırıq yoxdur.")
            return

        _gecikmə_nəticələr = gecikmə_riski_hesabla(tapşırıqlar)
        gecikmə_göstər("hamısı")

    # ── Overload riski düyməsi ───────────────────────
    def overload_yoxla(_):
        analiz_out.clear_output()
        gecikmə_out.clear_output()
        status_out.clear_output()
        formlar_box.children = []
        göndər_box.children = []
        with overload_out:
            overload_out.clear_output()
            print("👤 Jira-dan tapşırıqlar yüklənir...")

            tapşırıqlar, xəta = jira_tapşırıqları_çək()
            if xəta:
                print(f"❌ {xəta}")
                return
            if not tapşırıqlar:
                print("✅ Aktiv tapşırıq yoxdur.")
                return

            nəticələr = overload_riski_hesabla(tapşırıqlar)
            overload_out.clear_output()

        kartlar = [widgets.HTML(
            f"<h4 style='margin:8px 0'>👤 Resurs Overload — {len(nəticələr)} icraçı</h4>"
        )]
        for o in nəticələr:
            kartlar.append(overload_kartı_yarat(o))

        overload_out.clear_output()
        with overload_out:
            display(widgets.VBox(kartlar))

    gecikmə_btn.on_click(gecikmə_yoxla)
    overload_btn.on_click(overload_yoxla)

    # ── Status Report düyməsi ────────────────────────
    _status_nəticələr = []

    def status_göstər(filtr="hamısı"):
        if filtr == "hamısı":
            görünən = _status_nəticələr
        else:
            görünən = [t for t in _status_nəticələr if t["status"] == filtr]

        statuslar = sorted(set(t["status"] for t in _status_nəticələr))
        filtr_box = widgets.HBox([
            widgets.HTML("<b style='line-height:32px;margin-right:8px'>Filtr:</b>"),
            *[
                widgets.Button(
                    description=s,
                    button_style="primary" if s == filtr else "",
                    layout=widgets.Layout(height="32px",
                                         width=f"{max(100, len(s)*10)}px")
                )
                for s in ["hamısı"] + statuslar
            ]
        ], layout=widgets.Layout(margin="8px 0", flex_flow="row wrap"))

        for btn, s in zip(filtr_box.children[1:], ["hamısı"] + statuslar):
            def _click(_, s=s):
                status_göstər(s)
            btn.on_click(_click)

        say    = len(görünən)
        başlıq = widgets.HTML(
            f"<h4 style='margin:8px 0'>📊 Status Report — {say} tapşırıq</h4>"
        )

        if say == 0:
            məzmun = [başlıq, filtr_box,
                      widgets.HTML("<p style='color:#888'>Bu statusda tapşırıq yoxdur.</p>")]
        else:
            status_rəng = {
                "To Do":       "#3498db",
                "In Progress": "#f39c12",
                "In Review":   "#9b59b6",
                "Done":        "#27ae60",
            }
            kartlar = []
            for t in görünən:
                rəng     = status_rəng.get(t["status"], "#95a5a6")
                deadline = t["deadline"].strftime("%d.%m.%Y") if t["deadline"] else "—"
                sp       = t["story_points"] if t["story_points"] else "—"
                kartlar.append(widgets.HTML(
                    f"<div style='border-left:4px solid {rəng};padding:10px 14px;"
                    f"margin:4px 0;background:#fafafa;border-radius:4px;font-size:13px'>"
                    f"<div style='display:flex;justify-content:space-between'>"
                    f"<b>{t['key']} — {t['ad']}</b>"
                    f"<span style='background:{rəng};color:white;padding:2px 8px;"
                    f"border-radius:10px;font-size:11px'>{t['status']}</span></div>"
                    f"<div style='margin-top:5px;color:#555'>"
                    f"👤 <b>İcraçı:</b> {t['icraçı']} &nbsp;|&nbsp; "
                    f"📅 <b>Deadline:</b> {deadline} &nbsp;|&nbsp; "
                    f"📊 <b>Story Points:</b> {sp} &nbsp;|&nbsp; "
                    f"⚡ <b>Prioritet:</b> {t['prioritet']}</div>"
                    f"</div>"
                ))
            məzmun = [başlıq, filtr_box] + kartlar

        status_out.clear_output()
        with status_out:
            display(widgets.VBox(məzmun))

    def status_yoxla(_):
        nonlocal _status_nəticələr
        analiz_out.clear_output()
        gecikmə_out.clear_output()
        overload_out.clear_output()
        formlar_box.children = []
        göndər_box.children = []
        with status_out:
            status_out.clear_output()
            print("📊 Jira-dan tapşırıqlar yüklənir...")

        tapşırıqlar, xəta = jira_tapşırıqları_çək(done_daxil=True)
        if xəta:
            with status_out:
                status_out.clear_output()
                print(f"❌ {xəta}")
            return
        if not tapşırıqlar:
            with status_out:
                status_out.clear_output()
                print("✅ Aktiv tapşırıq yoxdur.")
            return

        _status_nəticələr = tapşırıqlar
        status_göstər("hamısı")

    status_btn.on_click(status_yoxla)

    # ── Analiz düyməsi ───────────────────────────────
    def analiz_et(_):
        gecikmə_out.clear_output()
        overload_out.clear_output()
        status_out.clear_output()
        with analiz_out:
            analiz_out.clear_output()
            göndər_box.children = []

            mətn = w_giriş.value.strip()
            if not mətn:
                print("⚠️  Zəhmət olmasa tapşırıq mətni yazın.")
                return

            sətirlər = [s.strip() for s in mətn.split("\n") if s.strip()]
            if not sətirlər:
                print("⚠️  Heç bir tapşırıq tapılmadı.")
                return

            print(f"🧠 {len(sətirlər)} sətir analiz edilir...")
            tasklar = ai_parse_çoxlu(sətirlər)
            if not tasklar:
                print("❌ Analiz alınmadı.")
                return

            analiz_out.clear_output()
            print(f"✅ {len(tasklar)} tapşırıq aşkar edildi.")

        formlar_box.children = []
        form_dəstləri = []
        kartlar       = []

        for i, task in enumerate(tasklar):
            ai_prioritet    = task.get("prioritet", "Medium")
            prioritet_səbəb = task.get("prioritet_səbəb", "")

            w_task = widgets.Text(
                value=task["task_adi"],
                description="📝 Task adı:",
                style=style, layout=layout
            )
            w_icraçı = widgets.Text(
                value=task["icraçı"],
                description="👤 İcraçı:",
                style=style, layout=layout
            )
            w_baslama = widgets.Text(
                value=bugun,
                description="📅 Başlama:",
                style=style, layout=layout,
                disabled=True
            )
            w_deadline = widgets.Text(
                value=task["deadline"],
                description="⏰ Bitmə tarixi:",
                style=style, layout=layout
            )
            w_story_points = widgets.BoundedIntText(
                value=task.get("story_points", 1),
                min=1, max=13,
                description="📊 Story Points:",
                style=style,
                layout=widgets.Layout(width="230px")
            )
            w_kpi = widgets.Textarea(
                value=task.get("kpi", ""),
                description="🎯 KPI:",
                placeholder="Mətnde 'hədəf' sözündən sonra yazılan hissə",
                style=style,
                layout=widgets.Layout(width="500px", height="80px")
            )
            w_prioritet = widgets.Dropdown(
                value=ai_prioritet,
                options=["High", "Medium", "Low"],
                description="⚡ Prioritet:",
                style=style,
                layout=widgets.Layout(width="290px")
            )
            w_daxil_et = widgets.Checkbox(
                value=True,
                description="Jira-ya əlavə et",
                style={"description_width": "initial"}
            )

            form_dəstləri.append({
                "w_task":         w_task,
                "w_icraçı":       w_icraçı,
                "w_deadline":     w_deadline,
                "w_story_points": w_story_points,
                "w_kpi":          w_kpi,
                "w_prioritet":    w_prioritet,
                "w_daxil_et":     w_daxil_et
            })

            kart_stil        = widgets.Layout(
                border="2px solid #e74c3c" if ai_prioritet == "High" else
                       "2px solid #f39c12" if ai_prioritet == "Medium" else
                       "1px solid #ddd",
                background_color="#fdecea" if ai_prioritet == "High" else "white",
                border_radius="8px", padding="12px", margin="0 0 10px 0"
            )
            xəbərdarlıq_rəng = "#e74c3c" if ai_prioritet == "High" else "#f39c12"

            kart = widgets.VBox(
                [
                    widgets.HTML(
                        f"<b style='font-size:13px'>📌 Tapşırıq {i+1}</b>"
                        f"<hr style='margin:5px 0'>"
                    ),
                    widgets.HTML(
                        f"<div style='background:{xəbərdarlıq_rəng};color:white;"
                        f"padding:6px 10px;border-radius:4px;margin-bottom:8px;font-size:12px'>"
                        f"⚡ AI Prioritet: <b>{ai_prioritet}</b><br>"
                        f"<i>{prioritet_səbəb}</i></div>"
                    ),
                    w_task, w_icraçı, w_baslama, w_deadline,
                    w_story_points, w_kpi, w_prioritet, w_daxil_et
                ],
                layout=kart_stil
            )
            kartlar.append(kart)

        formlar_box.children = kartlar

        göndər_btn = widgets.Button(
            description=f"✅  Jira-ya Göndər ({len(tasklar)} tapşırıq)",
            button_style="success",
            layout=widgets.Layout(width="300px", height="42px")
        )
        ləğv_btn = widgets.Button(
            description="❌  Ləğv et",
            button_style="danger",
            layout=widgets.Layout(width="130px", height="42px", margin="0 0 0 10px")
        )
        nəticə_out = widgets.Output()

        def göndər(_):
            with nəticə_out:
                nəticə_out.clear_output()

                # ── Məcburi sahələri yoxla ───────────────
                xətalı_kartlar = []
                for i, dəst in enumerate(form_dəstləri):
                    if not dəst["w_daxil_et"].value:
                        continue
                    boş = []
                    if not dəst["w_task"].value.strip():
                        boş.append("w_task")
                    if not dəst["w_icraçı"].value.strip():
                        boş.append("w_icraçı")
                    if not dəst["w_deadline"].value.strip():
                        boş.append("w_deadline")
                    if not dəst["w_kpi"].value.strip():
                        boş.append("w_kpi")
                    if dəst["w_story_points"].value < 1:
                        boş.append("w_story_points")

                    if boş:
                        xətalı_kartlar.append((i, boş))
                        for sahə in boş:
                            dəst[sahə].layout.border = "2px solid #e74c3c"
                            dəst[sahə].layout.padding = "2px"
                    else:
                        for sahə in ["w_task","w_icraçı","w_deadline","w_kpi","w_story_points"]:
                            dəst[sahə].layout.border = ""
                            dəst[sahə].layout.padding = ""

                if xətalı_kartlar:
                    adlar = {"w_task": "Task adı", "w_icraçı": "İcraçı",
                             "w_deadline": "Deadline", "w_kpi": "KPI",
                             "w_story_points": "Story Points"}
                    for i, boş in xətalı_kartlar:
                        sahə_adları = ", ".join(adnar := [adlar[s] for s in boş])
                        print(f"⚠️  Tapşırıq {i+1}: [{sahə_adları}] — boş qala bilməz")
                    print("
❌ Bütün məcburi sahələr doldurulmadan göndərmək olmaz.")
                    return

                uğurlu  = []
                keçildi = []
                xəta    = []

                for i, dəst in enumerate(form_dəstləri):
                    if not dəst["w_daxil_et"].value:
                        keçildi.append(i + 1)
                        continue

                    key = jira_issue_yarat(
                        task_adi     = dəst["w_task"].value,
                        icraçı       = dəst["w_icraçı"].value,
                        deadline     = dəst["w_deadline"].value,
                        prioritet    = dəst["w_prioritet"].value,
                        asılılıq     = None,
                        story_points = dəst["w_story_points"].value,
                        kpi          = dəst["w_kpi"].value
                    )

                    if key:
                        uğurlu.append((i + 1, key))
                    else:
                        xəta.append(i + 1)

                print(f"\n{'═'*42}")
                print(f"  📊 NƏTİCƏ:")
                print(f"  ✅ Uğurlu:  {len(uğurlu)} tapşırıq")
                if keçildi:
                    print(f"  ⏭️  Keçildi: {len(keçildi)} tapşırıq")
                if xəta:
                    print(f"  ❌ Xəta:    {len(xəta)} tapşırıq")
                for num, key in uğurlu:
                    print(f"     Tapşırıq {num}: {key} → {JIRA_URL}/browse/{key}")
                print(f"{'═'*42}")

        def ləğv(_):
            with nəticə_out:
                nəticə_out.clear_output()
                print("❌ Ləğv edildi. Heç nə yazılmadı.")

        göndər_btn.on_click(göndər)
        ləğv_btn.on_click(ləğv)

        göndər_box.children = [
            widgets.HTML("<hr style='margin:10px 0'>"),
            widgets.HBox([göndər_btn, ləğv_btn]),
            nəticə_out
        ]

    analiz_btn.on_click(analiz_et)

    display(widgets.HTML(
        "<h3 style='margin:0 0 10px 0'>🤖 AI Agent — Task Yaratma</h3>"
    ))
    display(widgets.VBox([
        w_giriş,
        widgets.HBox([analiz_btn, gecikmə_btn, overload_btn, status_btn],
                     layout=widgets.Layout(margin="8px 0 0 0")),
        analiz_out,
        gecikmə_out,
        overload_out,
        status_out,
        formlar_box,
        göndər_box
    ]))


# ═══════════════════════════════════════
# İŞLAT
# ═══════════════════════════════════════
agent()
